# *Setup*

In [2]:
%%time
import numpy as np
import pandas as pd

from hid_avan.eda import head_tail_info
from hid_avan.preprocessing import extract_arrays_from_daily_pr_dataset
from hid_avan.modeling import build_timeseries_df_from_daily_avg_pr
from hid_avan.Thiessen import thiessen

CPU times: total: 1.8 s
Wall time: 18.7 s


# Construindo a Série Temporal de Precipitações Médias Mensais

## Espacializando dados pontuais de precipitações médias diárias

In [3]:
_, coord_array, pr_array = extract_arrays_from_daily_pr_dataset(funceme_txt_path='../../data/raw/pr_daily_funceme_34750000_19730101_20260731.txt',
                                                                missing_pr_value=-999.)

In [4]:
%%time
avg_pr_array = thiessen(dados=pr_array.T,
                        lat=coord_array[:, 0],
                        lon=coord_array[:, 1],
                        pathshp='../../data/raw/34750000.asc',
                        pf=-1, sep=',', usenc=False)

CPU times: total: 766 ms
Wall time: 1min 59s


In [6]:
df = build_timeseries_df_from_daily_avg_pr(avg_pr_array=avg_pr_array,
                                           date_start='1973-01-01', date_end='2026-07-31',
                                           missing_pr_value=-999.)
#df.to_csv('../../data/processed/timeseries_daily_avg_pr.csv', index=False)
head_tail_info(df)

<class 'pandas.DataFrame'>
RangeIndex: 19570 entries, 0 to 19569
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    19570 non-null  datetime64[us]
 1   avg_pr  19569 non-null  float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 305.9 KB


,date,avg_pr
0,1973-01-01,NaN
1,1973-01-02,0.0
2,1973-01-03,0.0
3,1973-01-04,0.0
4,1973-01-05,0.3
19565,2026-07-27,0.0
19566,2026-07-28,0.0
19567,2026-07-29,0.0
19568,2026-07-30,0.0
19569,2026-07-31,0.0


None

## Agrupando em agregados mensais de precipitações médias

In [14]:
df = pd.read_csv('../../data/processed/timeseries_daily_avg_pr.csv', parse_dates=['date'])
df_agg = df.groupby(by=pd.Grouper(key='date', freq='MS'),
                    as_index=False).agg(agg_mean__avg_pr=('avg_pr', 'mean'))
#df_agg.to_csv('../../data/processed/aggregated-by-mean_timeseries_monthly_avg_pr.csv', index=False)
head_tail_info(df_agg)

<class 'pandas.DataFrame'>
RangeIndex: 643 entries, 0 to 642
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              643 non-null    datetime64[us]
 1   agg_mean__avg_pr  643 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 10.2 KB


,date,agg_mean__avg_pr
0,1973-01-01,1.483333
1,1973-02-01,4.489286
2,1973-03-01,8.745161
3,1973-04-01,8.866667
4,1973-05-01,4.729032
638,2026-03-01,5.241654
639,2026-04-01,4.343600
640,2026-05-01,1.518898
641,2026-06-01,0.303568
642,2026-07-01,0.073574


None